In [36]:
import os
import pickle
import warnings
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Вказуємо MLflow записувати логи в базу даних, щоб обійти баги Windows
mlflow.set_tracking_uri("sqlite:///mlflow.db")

warnings.filterwarnings('ignore')
print("Бібліотеки імпортовано успішно. Середовище готове до роботи.")

Бібліотеки імпортовано успішно. Середовище готове до роботи.


In [37]:
os.makedirs('data/raw', exist_ok=True)

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Визначаємо назви колонок для датасету
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]

# Зчитуємо дані, вказуючи, що заголовка немає (header=None), і передаємо наші назви
df = pd.read_csv(url, header=None, names=column_names, na_values='?')

# Видаляємо всі рядки, де є пропуски
df.dropna(inplace=True)

# Тепер колонка 'target' існує, і ми можемо з нею працювати
if df['target'].max() > 1:
    df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

df.to_csv('data/raw/heart_disease.csv', index=False)

print(f"Розмір датасету: {df.shape}")
print("Розподіл класів:")
print(df['target'].value_counts())
print("\nДатасет успішно збережено в 'data/raw/heart_disease.csv' для подальшого трекінгу через DVC.")

Розмір датасету: (297, 14)
Розподіл класів:
target
0    160
1    137
Name: count, dtype: int64

Датасет успішно збережено в 'data/raw/heart_disease.csv' для подальшого трекінгу через DVC.


In [38]:
def train_and_log_model(model_name, model, params=None, experiment_name="Heart_Disease_Variant9"):
    mlflow.set_experiment(experiment_name)
    
    with mlflow.start_run(run_name=model_name):
        if params:
            mlflow.log_params(params)
        else:
            mlflow.log_params(model.get_params())
            
        X = df.drop('target', axis=1)
        y = df['target']
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', model)
        ])
        
        cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
        mlflow.log_metric("cv_accuracy_mean", cv_scores.mean())
        mlflow.log_metric("cv_accuracy_std", cv_scores.std())
        
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='binary')
        rec = recall_score(y_test, y_pred, average='binary')
        f1 = f1_score(y_test, y_pred, average='binary')
        
        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_precision", prec)
        mlflow.log_metric("test_recall", rec)
        mlflow.log_metric("test_f1", f1)
        
        mlflow.sklearn.log_model(pipeline, "model")
        
        print(f"[{model_name}] Accuracy: {acc:.4f} | F1: {f1:.4f} | CV Mean: {cv_scores.mean():.4f}")
        
        return pipeline, acc

In [39]:
print("=== Експерименти з основною моделлю (Gradient Boosting) ===")

gb_params_list = [
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 150, "max_depth": 4, "learning_rate": 0.1}
]

for i, params in enumerate(gb_params_list, 1):
    model = GradientBoostingClassifier(random_state=42, **params)
    train_and_log_model(f"GradientBoosting_Exp{i}", model, params)

=== Експерименти з основною моделлю (Gradient Boosting) ===


2026/04/29 17:16:54 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/29 17:16:54 INFO mlflow.store.db.utils: Updating database tables
2026/04/29 17:16:56 INFO mlflow.tracking.fluent: Experiment with name 'Heart_Disease_Variant9' does not exist. Creating a new experiment.
2026/04/29 17:16:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/29 17:16:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[GradientBoosting_Exp1] Accuracy: 0.7667 | F1: 0.7407 | CV Mean: 0.7629


2026/04/29 17:17:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/29 17:17:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[GradientBoosting_Exp2] Accuracy: 0.8167 | F1: 0.8000 | CV Mean: 0.7798


2026/04/29 17:17:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/29 17:17:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[GradientBoosting_Exp3] Accuracy: 0.8333 | F1: 0.8148 | CV Mean: 0.7504


In [40]:
print("\n=== Експерименти з альтернативними моделями (SVC та MLPClassifier) ===")

svc_params_list = [
    {"C": 1.0, "kernel": "rbf"},
    {"C": 10.0, "kernel": "rbf"}
]

for i, params in enumerate(svc_params_list, 1):
    model = SVC(random_state=42, probability=True, **params)
    train_and_log_model(f"SVC_Exp{i}", model, params)

mlp_params_list = [
    {"hidden_layer_sizes": (100,), "max_iter": 500},
    {"hidden_layer_sizes": (50, 50), "max_iter": 500}
]

for i, params in enumerate(mlp_params_list, 1):
    model = MLPClassifier(random_state=42, **params)
    train_and_log_model(f"MLP_Exp{i}", model, params)


=== Експерименти з альтернативними моделями (SVC та MLPClassifier) ===


2026/04/29 17:17:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/29 17:17:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[SVC_Exp1] Accuracy: 0.8500 | F1: 0.8302 | CV Mean: 0.8051


2026/04/29 17:17:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/29 17:17:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[SVC_Exp2] Accuracy: 0.8333 | F1: 0.8148 | CV Mean: 0.7716


2026/04/29 17:17:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/29 17:17:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[MLP_Exp1] Accuracy: 0.8500 | F1: 0.8364 | CV Mean: 0.7842


2026/04/29 17:17:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/29 17:17:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[MLP_Exp2] Accuracy: 0.8500 | F1: 0.8421 | CV Mean: 0.7717


In [41]:
os.makedirs('models', exist_ok=True)

best_params = {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1}
best_model = GradientBoostingClassifier(random_state=42, **best_params)

print("\n=== Тренування та збереження найкращої моделі ===")
best_pipeline, best_acc = train_and_log_model("Best_GradientBoosting", best_model, best_params)

model_path = 'models/best_heart_pipeline.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(best_pipeline, f)

print(f"\nНайкращу модель успішно збережено за шляхом: {model_path}")


=== Тренування та збереження найкращої моделі ===


2026/04/29 17:17:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/29 17:17:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Best_GradientBoosting] Accuracy: 0.7667 | F1: 0.7407 | CV Mean: 0.7629

Найкращу модель успішно збережено за шляхом: models/best_heart_pipeline.pkl
